In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import scipy.io as sio
from torch.utils.data import Dataset, DataLoader, random_split

In [2]:
input_data = sio.loadmat('input_file500_50hDS.mat')
output_data = sio.loadmat('output_file500_50hDS.mat')

X1 = input_data['x1']
X2 = input_data['x2']
Y  = output_data['Y']

print(X1.shape, X2.shape, Y.shape)

(500, 2000, 1000) (500, 2000, 1000) (500, 4)


In [3]:
def extract_peak(x):
    row, col = np.unravel_index(np.argmax(x), x.shape)
    return row, col

features = []

for i in range(len(X1)):
    r1, c1 = extract_peak(X1[i])
    r2, c2 = extract_peak(X2[i])

    features.append([c1, r1, c2, r2])

features = np.array(features).astype(np.float32)


In [4]:
# Normalize features
features = (features - np.mean(features, axis=0)) / (np.std(features, axis=0) + 1e-6)

# Normalize outputs
Y = Y.astype(np.float32)

Y[:, 0:2] /= 1000.0   # position
Y[:, 2:4] /= 100.0    # velocity

In [5]:
class PeakDataset(torch.utils.data.Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.Y[idx])

dataset = PeakDataset(features, Y)

In [6]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=16)

In [7]:
class PeakNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(4, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 4)
        )

    def forward(self, x):
        return self.net(x)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PeakNet().to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [15]:
epochs = 200

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        pred = model(x_batch)
        loss = criterion(pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # validation
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            pred = model(x_batch)
            val_loss += criterion(pred, y_batch).item()

    print(f"Epoch {epoch+1:02d} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

Epoch 01 | Train: 0.0039 | Val: 0.0014
Epoch 02 | Train: 0.0042 | Val: 0.0018
Epoch 03 | Train: 0.0041 | Val: 0.0013
Epoch 04 | Train: 0.0039 | Val: 0.0012
Epoch 05 | Train: 0.0037 | Val: 0.0015
Epoch 06 | Train: 0.0038 | Val: 0.0017
Epoch 07 | Train: 0.0039 | Val: 0.0013
Epoch 08 | Train: 0.0039 | Val: 0.0016
Epoch 09 | Train: 0.0037 | Val: 0.0012
Epoch 10 | Train: 0.0035 | Val: 0.0016
Epoch 11 | Train: 0.0038 | Val: 0.0014
Epoch 12 | Train: 0.0038 | Val: 0.0012
Epoch 13 | Train: 0.0034 | Val: 0.0011
Epoch 14 | Train: 0.0039 | Val: 0.0019
Epoch 15 | Train: 0.0039 | Val: 0.0013
Epoch 16 | Train: 0.0038 | Val: 0.0011
Epoch 17 | Train: 0.0038 | Val: 0.0011
Epoch 18 | Train: 0.0035 | Val: 0.0010
Epoch 19 | Train: 0.0034 | Val: 0.0011
Epoch 20 | Train: 0.0036 | Val: 0.0011
Epoch 21 | Train: 0.0036 | Val: 0.0011
Epoch 22 | Train: 0.0044 | Val: 0.0012
Epoch 23 | Train: 0.0039 | Val: 0.0011
Epoch 24 | Train: 0.0033 | Val: 0.0010
Epoch 25 | Train: 0.0034 | Val: 0.0013
Epoch 26 | Train: 0.0036 

In [16]:
errors_pos = []
errors_vel = []

model.eval()

with torch.no_grad():
    for x_sample, y_true in dataset:
        pred = model(x_sample.unsqueeze(0).to(device)).cpu().numpy()[0]

        # de-normalize
        pred[0:2] *= 1000
        pred[2:4] *= 100

        y_true = y_true.numpy()
        y_true[0:2] *= 1000
        y_true[2:4] *= 100

        pos_error = np.linalg.norm(pred[0:2] - y_true[0:2])
        vel_error = np.linalg.norm(pred[2:4] - y_true[2:4])

        errors_pos.append(pos_error)
        errors_vel.append(vel_error)

print("\n==== PEAK MODEL PERFORMANCE ====")
print(f"Avg Position Error: {np.mean(errors_pos):.2f} m")
print(f"Avg Velocity Error: {np.mean(errors_vel):.2f} m/s")


==== PEAK MODEL PERFORMANCE ====
Avg Position Error: 14.32 m
Avg Velocity Error: 0.31 m/s
